In [ ]:
import os
import random
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from ase import units
from ase.build import bulk
from ase.neighborlist import neighbor_list
from ase.io import read, write
from ase.optimize import BFGS
from ase.md.langevin import Langevin
from ase.md.velocitydistribution import MaxwellBoltzmannDistribution

from fairchem.core import FAIRChemCalculator
from fairchem.core import pretrained_mlip
# os.environ["HF_TOKEN"] = "[YOUR TOKEN HERE]"
# !huggingface-cli login --token $HF_TOKEN

import h5py

### MC

In [ ]:
SUPERCELL_SIZE = (10, 10, 10)
MC_STEPS = 14000000 # num atoms * (num frames + equilibration steps) --> in this case 2,000 atoms * (5*1000 frames + 5*400 equilibration steps)
T = 4.2 # in relative units, should be comparable to J and nearest neighbors (delta E)
J = 1 # Ising model where we favor Fe-Al over Fe-Fe or Al-Al (ordering tendency, J > 0)
index = int(os.environ.get("MC_INDEX", 1))

# output directory structure: fe-al/{mc, relaxation, md}, each with
#   final_structures/  -- last frame of that part of the run, one per index
#   *_diffuse/         -- save_to_yell output averaged over ALL runs (filled by run_pipeline.py, not this notebook)
#   *-output/          -- extra outputs (energies, trajectories, logs, etc.)
mc_dir = "additional/fe-al-10/fe-al_mc_structures"
relax_dir = "additional/fe-al-10/fe-al_relaxed_structures"
md_dir = "additional/fe-al-10/fe-al_md_structures"

mc_diffuse_dir = os.path.join(mc_dir, "average_diffuse")
relaxed_diffuse_dir = os.path.join(relax_dir, "average_diffuse")
md_diffuse_dir = os.path.join(md_dir, "average_diffuse")

for d in [mc_diffuse_dir, relaxed_diffuse_dir, md_diffuse_dir,
          os.path.join(mc_dir, "final_structures"),
          os.path.join(relax_dir, "final_structures"),
          os.path.join(md_dir, "final_structures")]:
    os.makedirs(d, exist_ok=True)

In [ ]:
# create Fe-Al structure
a_lattice = 2.867
atoms = bulk("Fe", "bcc", a=a_lattice, cubic=True) # create Fe unit cell
atoms = atoms.repeat(SUPERCELL_SIZE) # create supercell
num_atoms = len(atoms)

# create random 80-20 distribution of Fe and Al
num_al = int(0.20 * num_atoms)
num_fe = num_atoms - num_al
species = np.array([0] * num_fe + [1] * num_al)
np.random.shuffle(species)

# set up nearest neighbor matrix for BCC structure
bcc_nn_distance = a_lattice * np.sqrt(3) / 2
cutoff_distance = 1.15 * bcc_nn_distance # slightly larger than the nearest neighbor distance to ensure we capture all neighbors
i, j = neighbor_list('ij', atoms, cutoff_distance) # neighbor_list returns pairs of indices (i, j) where atoms i and j are neighbors within the cutoff distance
nn_matrix = np.zeros((num_atoms, num_atoms), dtype=int)
nn_matrix[i, j] = 1 # set nn_matrix[i, j] = 1 if atoms i and j are neighbors

neighbor_lookup = [np.where(nn_matrix[i] > 0)[0] for i in range(num_atoms)] # create a lookup table where neighbor_lookup[i] gives the indices of the neighbors of atom i

In [ ]:
energy_history = []
trajectory_data = []
current_energy = 0

# calculate initial energy of the random configuration
# loops over the upper triangle to avoid double counting
rows, cols = np.where(np.triu(nn_matrix)) # get indices of nearest neighbor pairs
for i, j in zip(rows, cols):
    if species[i] != species[j]:
        current_energy -= J  # decrease energy for Fe-Al bonds
    else:
        current_energy += J  # increased energy for Fe-Fe or Al-Al bonds
    
# monte carlo loop
for step in range(MC_STEPS):
    idx_a = random.randint(0, num_atoms - 1)
    idx_b = random.randint(0, num_atoms - 1)
    while idx_a == idx_b:
        idx_b = random.randint(0, num_atoms - 1)
    
    # only perform calculation if we have different elements
    if species[idx_a] != species[idx_b]:
        s_a = species[idx_a]
        s_b = species[idx_b]
        
        # use lookup table to find indices of nearest neighbors
        neighbors_a = neighbor_lookup[idx_a]
        neighbors_b = neighbor_lookup[idx_b]
        
        # calculate local dE based on nearest neighbors (first around atom A, then around atom B)
        dE = 0.0
        for j in neighbors_a:
            if j == idx_b:
                continue # skip atom A-B bond since energy is the same before and after the swap
            if species[j] == s_a:
                dE -= 2 * J  # like-pair becomes unlike-pair (decrease energy)
            else:
                dE += 2 * J  # unlike-pair becomes like-pair (increase energy)
        for j in neighbors_b:
            if j == idx_a:
                continue # skip atom B-A bond
            if species[j] == s_b:
                dE -= 2 * J  # like-pair becomes unlike-pair
            else:
                dE += 2 * J  # unlike-pair becomes like-pair
        
        # Metropolis acceptance criterion
        if dE < 0 or random.random() < np.exp(-dE / T):
            # if accepted, perform the swap and update current energy
            species[idx_a], species[idx_b] = s_b, s_a
            current_energy += dE
            
    # track energy changes
    energy_history.append(current_energy)

    # saves energy and chemical configuration every 5 whole structure swaps (10,000 steps since 2,000 atoms)
    if (step + 1) % 10000 == 0: # use step + 1 so we don't save inital configuration
            current_elements = ['Al' if x == 1 else 'Fe' for x in species]
            trajectory_data.append({
                "Step": step + 1,
                "Total_Energy_eV": current_energy,
                "Chemical_Configuration": ",".join(current_elements)
            })

# update atoms object with final chemical configuration
final_symbols = ['Al' if x == 1 else 'Fe' for x in species]
atoms.set_chemical_symbols(final_symbols)

# save final MC configuration (last frame of the MC run)
write(os.path.join(mc_dir, "final_structures", f"mc_final_{index:03d}.cif"), atoms)


### Relaxation

In [ ]:
# relax structure
UMA_MODEL = "uma-s-1p1"
UMA_TASK = "omat" # task for bulk materials
seed = np.random.randint(0, np.iinfo(np.int32).max, dtype=int)
predictor = pretrained_mlip.get_predict_unit(UMA_MODEL, device="cuda", seed=seed)
calculator = FAIRChemCalculator(predictor, task_name=UMA_TASK)

atoms_relaxed = atoms.copy()
atoms_relaxed.calc = calculator

opt = BFGS(atoms_relaxed, trajectory=None)
opt.run(fmax=0.01)

final_frame = atoms_relaxed.copy()
os.makedirs(os.path.join(relax_dir, "final_structures"), exist_ok=True)
final_frame.write(os.path.join(relax_dir, "final_structures", f"relaxed_frame_{index:03d}.cif"))

### MD

In [ ]:
TEMPERATURE_K = 300.0
TIME_STEP = 5.0 * units.fs
FRICTION = 0.002 / units.fs # thermostat coupling strength -- lower value means atoms don't feel heat bath (more natural dynamics but slower equilibration)
NUM_STEPS = 3000 # TIME_STEP*NUM_STEPS = 15 ps
LOG_INTERVAL = 200 # number of frames = NUM_STEPS / LOG_INTERVAL

In [ ]:
# set up MD simulation, starting from the final relaxed structure
md_atoms = final_frame.copy()
md_atoms.calc = calculator

MaxwellBoltzmannDistribution(md_atoms, temperature_K=TEMPERATURE_K) # assign initial velocities based on Maxwell-Boltzmann distribution

dyn = Langevin(
    md_atoms,
    timestep=TIME_STEP,
    temperature_K=TEMPERATURE_K,
    friction=FRICTION,
    trajectory=None,
    logfile=None,
    loginterval=LOG_INTERVAL
)

# track energy history
step_history = []
epot_history = []
ekin_history = []
temp_history = []

In [ ]:
# function to save + print MD status
def print_md_status():
    step = dyn.get_number_of_steps()
    epot = md_atoms.get_potential_energy() / len(md_atoms) 
    ekin = md_atoms.get_kinetic_energy() / len(md_atoms) 
    temp = md_atoms.get_temperature() 
    
    # save values
    step_history.append(step)
    epot_history.append(epot * len(md_atoms)) # total eV
    ekin_history.append(ekin * len(md_atoms)) # total eV
    temp_history.append(temp) # K
    
    print(f"Step: {step:4d} | "
          f"Temp: {temp:6.1f} K | "
          f"E_pot: {epot:.4f} eV/atom | "
          f"E_kin: {ekin:.4f} eV/atom")
    
# run simulation
dyn.attach(print_md_status, interval=LOG_INTERVAL)
dyn.run(NUM_STEPS)

# save final MD structure (last frame of the MD run)
trajectory_frames = read(traj_filename, index=':')
trajectory_frames[-1].write(os.path.join(md_dir, "final_structures", f"md_final_{index:03d}.cif"))

### Plots

In [ ]:
# hk0 slices of the aggregated diffuse patterns
# these h5 files are produced by run_pipeline.py AFTER all indices have finished,
# so this cell just skips anything that hasn't been computed yet
plot_targets = [
    ("MC", mc_diffuse_dir),
    ("Relaxed", relaxed_diffuse_dir),
    ("MD", md_diffuse_dir),
]

for label, diffuse_dir in plot_targets:
    h5_filename = os.path.join(diffuse_dir, "diffuse_symm.h5")
    if not os.path.exists(h5_filename):
        print(f"{h5_filename} not found -- run run_pipeline.py (or --average-only) first")
        continue

    with h5py.File(h5_filename, "r") as f:
        data_3d = f["data"][:]
        lower_limits = f["lower_limits"][:]  
        step_sizes = f["step_sizes"][:]      

    # setting up coordinates axes from reciprocal space data
    shape_h, shape_k, shape_l = data_3d.shape
    h_axis = lower_limits[0] + np.arange(shape_h) * step_sizes[0]
    k_axis = lower_limits[1] + np.arange(shape_k) * step_sizes[1]
    l_axis = lower_limits[2] + np.arange(shape_l) * step_sizes[2]

    l_center_idx = data_3d.shape[2] // 2  # midpoint of L axis corresponds to L=0 plane
    hk0_slice = data_3d[:, :, l_center_idx]

    plt.figure(figsize=(7, 6))
    plt.imshow(
        hk0_slice, 
        cmap='Greys', 
        origin='lower',
        extent=[h_axis.min(), h_axis.max(), k_axis.min(), k_axis.max()],
        aspect='equal',
        vmin=0, vmax=np.max(hk0_slice) * 0.0001
    )
    plt.colorbar(label='Diffuse Scattering Intensity')
    plt.title(f'Fe-Al Diffuse Scattering ({label}, Ensemble Average)\n(HK0) Plane')
    plt.xlabel('$H$ (r.l.u.)')
    plt.ylabel('$K$ (r.l.u.)')
    plt.xticks(np.arange(-4, 5, 2))
    plt.yticks(np.arange(-4, 5, 2))
    plt.tight_layout()
    plt.show()